In [1]:
from torch_geometric.datasets import Amazon
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/Users/zhangyue/Workspace/github/learning-toy-examples/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

In [ ]:
# classifying the products into different categories
dataset = Amazon(root='data/amazon', name='Computers')

print(dataset)
print(dataset[0])
print(dataset[0].x.shape)
print(dataset[0].y.shape)
print(dataset[0].edge_index.shape)

AmazonComputers()
Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])
torch.Size([13752, 767])
torch.Size([13752])
torch.Size([2, 491722])


In [3]:
np.unique(dataset[0].y)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

# Model

In [14]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import GCNConv
from sklearn.model_selection import StratifiedKFold

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        return self.conv2(x, edge_index)


def run_epoch(model, optimizer, data, train_mask, val_mask):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    train_loss = F.cross_entropy(out[train_mask], data.y[train_mask])
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        val_loss = F.cross_entropy(out[val_mask], data.y[val_mask])
    return train_loss.item(), val_loss.item()


@torch.no_grad()
def masked_accuracy(model, data, mask):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out[mask].argmax(dim=-1)
    return (pred == data.y[mask]).float().mean().item()


data = dataset[0].to(device)
num_nodes = data.num_nodes
num_classes = int(data.y.max().item()) + 1
in_channels = data.num_node_features
hidden_channels = 64
max_epochs = 500
patience = 20  # early stop after this many epochs without val improvement
min_delta = 1e-4
lr = 0.005
weight_decay = 5e-4  # L2 penalty in Adam; works with dropout against overfitting
# Reduce LR when val loss stalls; stop training only after more patience (see loop below)
lr_scheduler_patience = 15
lr_factor = 0.5
min_lr = 1e-5
n_folds = 5

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
fold_history = {}

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.arange(num_nodes), data.y.cpu().numpy())):
    train_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
    train_mask[train_idx] = True
    val_mask[val_idx] = True

    model = GCN(in_channels, hidden_channels, num_classes).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_factor,
        patience=lr_scheduler_patience,
        min_lr=min_lr,
        threshold=min_delta,
    )

    train_losses, val_losses, lr_history = [], [], []
    best_val = float("inf")
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(max_epochs):
        train_loss, val_loss = run_epoch(model, optimizer, data, train_mask, val_mask)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        lr_history.append(optimizer.param_groups[0]["lr"])

        if val_loss < best_val - min_delta:
            best_val = val_loss
            epochs_without_improvement = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    train_acc = masked_accuracy(model, data, train_mask)
    val_acc = masked_accuracy(model, data, val_mask)

    fold_history[f"fold_{fold_idx}"] = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "lr_history": lr_history,
        "best_val_loss": best_val,
        "epochs_trained": len(train_losses),
        "final_lr": lr_history[-1],
        "train_acc": train_acc,
        "val_acc": val_acc,
    }
    print(
        f"fold {fold_idx}: stopped at epoch {len(train_losses)}/{max_epochs}, "
        f"best val_loss={best_val:.4f}, "
        f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}, "
        f"final_lr={lr_history[-1]:.2e}"
    )

train_accs = [fold_history[f"fold_{i}"]["train_acc"] for i in range(n_folds)]
val_accs = [fold_history[f"fold_{i}"]["val_acc"] for i in range(n_folds)]
print(
    f"\nCV summary: train_acc={np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}, "
    f"val_acc={np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}"
)

# things to check
# - overfitting: train_loss < val_loss? if overfitting, consider dropouts + weight_decay(L2 regularization, encourages smaller weights)
# - learning rate: too high or too low? LR scheduler for plateau
# - number of epochs: too many or too few?


fold 0: stopped at epoch 426/500, best val_loss=0.3126, train_acc=0.9261, val_acc=0.9088, final_lr=1.25e-03
fold 1: stopped at epoch 500/500, best val_loss=0.3325, train_acc=0.9261, val_acc=0.9149, final_lr=2.50e-03
fold 2: stopped at epoch 443/500, best val_loss=0.2998, train_acc=0.9230, val_acc=0.9073, final_lr=2.50e-03
fold 3: stopped at epoch 449/500, best val_loss=0.3199, train_acc=0.9280, val_acc=0.9084, final_lr=5.00e-03
fold 4: stopped at epoch 383/500, best val_loss=0.3192, train_acc=0.9286, val_acc=0.9069, final_lr=2.50e-03

CV summary: train_acc=0.9264 ± 0.0019, val_acc=0.9092 ± 0.0029


In [15]:
# todo
# - edge classification.
# - variants with batched size.